# Cifrado de imagen

Dos etapas:

1. **Difusion**: cada canal (R, G, B) se agrupa en bloques de 4 valores y se
   multiplica en cadena por matrices 4x4 construidas con matrices de Pauli.
   La llave (una lista de indices 1-24) selecciona que matriz se usa en cada
   paso de la cadena.
2. **Permutacion**: la parte real del resultado de la difusion se convierte a
   bits y se reordena fila por fila con un automata celular de 8 bits. Que
   conjunto de reglas usa cada fila sale de la misma secuencia caotica que
   genero la llave.

Las funciones de cada etapa estan en `difusion_pauli.py` y
`permutacion_automata.py` (mismo directorio que este notebook); la llave sale
de `caos.py`. Este notebook solo orquesta el flujo.

In [ ]:
%pip install numpy pillow matplotlib tifffile

In [ ]:
from pathlib import Path
import numpy as np
import tifffile
from PIL import Image

from caos import generar_llave
from difusion_pauli import (
    generar_matrices_4x4, matrices_encriptacion, obtener_rgb_como_bloques,
    cifrar_imagen, canal_a_2d, separar_en_tres_capas,
)
from permutacion_automata import generar_indices_reglas, cifrar_filas_automata
from io_imagen import (
    obtener_rgb, visualizar_canal_color, canal_a_filas_binarias,
    filas_a_matriz_pixeles, normalizar,
)

In [ ]:
# Rutas: este notebook vive un nivel por debajo de la raiz del proyecto,
# junto a Imagenes/ y Resultados/
base = Path().resolve().parent
ruta_imagen = base / "Imagenes" / "Prueba1.jpg"

carpeta_resultados = base / "Resultados"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

ruta_cifrado = carpeta_resultados / "Img Cifrada.tiff"
ruta_meta = carpeta_resultados / "cifrado_meta.txt"

## Cargar la imagen y ver sus canales

In [ ]:
R, G, B = obtener_rgb(str(ruta_imagen))
h, w = R.height, R.width

display(visualizar_canal_color(R, 'R'))
display(visualizar_canal_color(G, 'G'))
display(visualizar_canal_color(B, 'B'))

## Generar la llave a partir de la imagen

In [ ]:
key, x0, iteraciones_usadas = generar_llave(R, G, B, verbose=True)

## Difusion (matrices de Pauli)

In [ ]:
R_b, G_b, B_b = obtener_rgb_como_bloques(R, G, B)

matrices_rotadas = generar_matrices_4x4()
encriptacion = matrices_encriptacion(key, matrices_rotacion=matrices_rotadas)

c_r, c_g, c_b = cifrar_imagen(R_b, G_b, B_b, matrices_finales=encriptacion)

# Volver a las dimensiones (h, w) de la imagen original
c_r_2d = canal_a_2d(c_r, h, w)
c_g_2d = canal_a_2d(c_g, h, w)
c_b_2d = canal_a_2d(c_b, h, w)

# Separar cada canal complejo en magnitud real, magnitud imaginaria y mapa
# de signos: son las tres "capas" que se guardan por separado
r_real, r_imag, r_signos = separar_en_tres_capas(c_r_2d)
g_real, g_imag, g_signos = separar_en_tres_capas(c_g_2d)
b_real, b_imag, b_signos = separar_en_tres_capas(c_b_2d)

cifrado_stack = np.stack([
    r_real, g_real, b_real,       # Magnitudes reales
    r_imag, g_imag, b_imag,       # Magnitudes imaginarias
    r_signos, g_signos, b_signos  # Mapas de signos
], axis=0)

np.set_printoptions(suppress=True, precision=2)
print(cifrado_stack)

## Permutacion (automata celular)

Solo se permuta la magnitud real (`r_real`, `g_real`, `b_real`); la parte
imaginaria y el mapa de signos pasan sin tocar a la etapa de guardado. Para
poder cifrarla con el automata, la magnitud real se convierte primero a bits.

In [ ]:
# Bits necesarios por pixel: el valor mas grande entre los 3 canales define
# cuantos bits hacen falta para representarlo
max_val = int(np.max([r_real.max(), g_real.max(), b_real.max()]))
num_bits = max_val.bit_length()

R_int = r_real.astype(int)
G_int = g_real.astype(int)
B_int = b_real.astype(int)

filas_R = canal_a_filas_binarias(R_int, num_bits)
filas_G = canal_a_filas_binarias(G_int, num_bits)
filas_B = canal_a_filas_binarias(B_int, num_bits)

# La secuencia de reglas continua el mapa logistico justo donde lo dejo la
# generacion de la llave (iteraciones_usadas)
reglas = generar_indices_reglas(x0, h, iteraciones_ignorar=iteraciones_usadas)

filas_R_cif = cifrar_filas_automata(filas_R, reglas)
filas_G_cif = cifrar_filas_automata(filas_G, reglas)
filas_B_cif = cifrar_filas_automata(filas_B, reglas)

print(f"Valor maximo: {max_val}")
print(f"Bits por pixel: {num_bits}")
print(f"Longitud de la secuencia de reglas: {len(reglas)}")

## Guardar resultado

El TIFF de salida tiene 4 paginas:

1. Version normalizada a 0-255 de la magnitud real ya cifrada (solo para
   poder abrir el archivo y ver algo con un visor comun).
2. La magnitud real cifrada por el automata, tal como se necesita para
   descifrar (`filas_*_cif`, con el padding a multiplo de 8 incluido).
3. La magnitud imaginaria, sin tocar por el automata.
4. El mapa de signos, sin tocar por el automata.

`cifrado_meta.txt` guarda la llave, `x0`, `iteraciones_usadas` y `num_bits`.
Este ultimo dato antes no se guardaba y decrypt.ipynb lo reconstruia
adivinando a partir del ancho de fila con padding; guardarlo directamente
evita esa heuristica.

In [ ]:
# Reusa la misma funcion de io_imagen que usa decrypt.ipynb, solo para
# poder normalizar y visualizar la pagina 0 del TIFF
R_cif_int = filas_a_matriz_pixeles(filas_R_cif, num_bits, w)
G_cif_int = filas_a_matriz_pixeles(filas_G_cif, num_bits, w)
B_cif_int = filas_a_matriz_pixeles(filas_B_cif, num_bits, w)

with tifffile.TiffWriter(str(ruta_cifrado)) as tif:
    R_vis = normalizar(R_cif_int)
    G_vis = normalizar(G_cif_int)
    B_vis = normalizar(B_cif_int)
    visual_rgb = np.stack([R_vis, G_vis, B_vis], axis=-1)
    tif.write(visual_rgb, photometric='rgb')

    for filas_cif in (filas_R_cif, filas_G_cif, filas_B_cif):
        arr_bits = np.array(filas_cif, dtype=np.uint8)
        tif.write(arr_bits, photometric='minisblack')

    for capa in (r_imag, g_imag, b_imag):
        tif.write(capa.astype(np.float32), photometric='minisblack')

    for capa in (r_signos, g_signos, b_signos):
        tif.write(capa.astype(np.float32), photometric='minisblack')

pagina0 = tifffile.imread(str(ruta_cifrado), key=0)
display(Image.fromarray(pagina0, mode='RGB'))

with open(str(ruta_meta), "w") as f:
    f.write(f"Llave={key}\n")
    f.write(f"x0={x0}\n")
    f.write(f"iteraciones_usadas={iteraciones_usadas}\n")
    f.write(f"num_bits={num_bits}\n")